In [5]:
import scanpy as sc

In [15]:
import re

def make_valid_names(names_vector):
    """
    Convert drug names to valid Python variable names by replacing invalid characters
    and ensuring the name starts with a letter or underscore.
    """
    valid_names = []
    for name in names_vector:
        # Replace invalid characters with underscores
        cleaned = re.sub(r'[^0-9a-zA-Z_]', '_', name)
        
        # Ensure the name starts with a letter or underscore
        if not re.match(r'^[a-zA-Z_]', cleaned):
            cleaned = f'X_{cleaned}'
        
        valid_names.append(cleaned)
    return valid_names

def clean_column_names(names_vector):
    """
    Remove 'factor(valid_drug_names)X' prefix and replace multiple underscores with a single underscore.
    """
    cleaned_names = []
    for name in names_vector:
        # Remove specific prefix
        name = re.sub(r'^factor\(valid_drug_names\)X', '', name)
        
        # Replace multiple underscores with a single underscore
        name = re.sub(r'_+', '_', name)
        
        cleaned_names.append(name)
    return cleaned_names

def clean_contrast_names(names_vector):
    """
    Clean contrast names by removing leading underscores and ensuring valid variable names.
    """
    cleaned_names = []
    for name in names_vector:
        # Remove leading underscores
        name = re.sub(r'^_+', '', name)
        
        # Ensure it starts with a letter or underscore
        if not re.match(r'^[a-zA-Z_]', name):
            name = f'X_{name}'
        
        # Replace non-alphanumeric characters with underscores
        name = re.sub(r'[^0-9a-zA-Z_]', '_', name)
        
        cleaned_names.append(name)
    return cleaned_names

In [1]:
# Tahoe 100m

In [3]:
file0 = '/lustre/groups/ml01/workspace/manuel.gander/data/sub_adatas/bulked/'

In [6]:
adata = sc.read_h5ad(file0+'all_comb.h5ad')

In [13]:
conditions = sorted(set(adata.obs['drugname_drugconc']))

In [23]:
valid_names = make_valid_names(conditions)
column_cleaned = clean_column_names(valid_names)
contrast_cleaned = clean_contrast_names(column_cleaned)

In [29]:
D_exp = dict(zip(conditions, contrast_cleaned))

In [34]:
D_exp_inv = dict(zip(contrast_cleaned, conditions))

In [52]:
D_exp_inv["X18β_Glycyrrhetinic_acid_0_05_uM_"] = "[('18β-Glycyrrhetinic acid', 0.05, 'uM')]"
D_exp_inv["X18β_Glycyrrhetinic_acid_0_5_uM_"] = "[('18β-Glycyrrhetinic acid', 0.5, 'uM')]"
D_exp_inv["X18β_Glycyrrhetinic_acid_5_0_uM_"] = "[('18β-Glycyrrhetinic acid', 5.0, 'uM')]"


D_exp_inv["X5_Fluorouracil_5_0_uM_"] = "[('5-Fluorouracil', 5.0, 'uM')]"
D_exp_inv["X5_Fluorouracil_0_5_uM_"] = "[('5-Fluorouracil', 0.5, 'uM')]"
D_exp_inv["X5_Fluorouracil_0_05_uM_"] = "[('5-Fluorouracil', 0.05, 'uM')]"


D_exp_inv["γ_Oryzanol_5_0_uM_"] = "[('γ-Oryzanol', 5.0, 'uM')]"
D_exp_inv["γ_Oryzanol_0_5_uM_"] = "[('γ-Oryzanol', 0.5, 'uM')]"
D_exp_inv["γ_Oryzanol_0_05_uM_"] = "[('γ-Oryzanol', 0.05, 'uM')]"


D_exp_inv["X9_ING_41_5_0_uM_"] = "[('9-ING-41', 5.0, 'uM')]"
D_exp_inv["X9_ING_41_0_5_uM_"] = "[('9-ING-41', 0.5, 'uM')]"
D_exp_inv["X9_ING_41_0_05_uM_"] = "[('9-ING-41', 0.05, 'uM')]"

D_exp_inv["X8_Hydroxyquinoline_5_0_uM_"] = "[('8-Hydroxyquinoline', 5.0, 'uM')]"
D_exp_inv["X8_Hydroxyquinoline_0_5_uM_"] = "[('8-Hydroxyquinoline', 0.5, 'uM')]"
D_exp_inv["X8_Hydroxyquinoline_0_05_uM_"] = "[('8-Hydroxyquinoline', 0.05, 'uM')]"

D_exp_inv["X5_Azacytidine_5_0_uM_"] = "[('5-Azacytidine', 5.0, 'uM')]"
D_exp_inv["X5_Azacytidine_0_5_uM_"] = "[('5-Azacytidine', 0.5, 'uM')]"
D_exp_inv["X5_Azacytidine_0_05_uM_"] = "[('5-Azacytidine', 0.05, 'uM')]"

D_exp_inv["X4EGI_1_5_0_uM_"] = "[('4EGI-1', 5.0, 'uM')]"
D_exp_inv["X4EGI_1_0_5_uM_"] = "[('4EGI-1', 0.5, 'uM')]"
D_exp_inv["X4EGI_1_0_05_uM_"] = "[('4EGI-1', 0.05, 'uM')]"

In [2]:
import pandas as pd
cellline = "A549"
output_file = f"/lustre/groups/ml01/workspace/manuel.gander/data/postlimma/mDf2/{cellline}_differential_expression_results.parquet"
df0 = pd.read_parquet(output_file)

In [53]:
df0['perturbation'] = df0.condition.map(D_exp_inv)

In [54]:
df0['perturbation'] = [a if 'plate' in a else b for a,b in zip(df0.condition, df0.perturbation)]

In [63]:
df = pd.DataFrame(D_exp_inv, index=range(1)).T

In [64]:
df.to_csv('Experiment_dict_Tahoe100m.csv')